# 1. What you'll learn

        - inspect real FashionMNIST images and tensor shapes
- train a tiny CPU convolutional network
- destroy spatial structure by shuffling pixels

        **The one question this notebook answers:** How do shared local filters learn image patterns that survive small shifts?

# 2. Setup

This lesson keeps cnn image classification small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import torch
from torch import nn
from torch.utils.data import DataLoader,Subset,TensorDataset
from torchvision import datasets,transforms
from sklearn.metrics import accuracy_score,confusion_matrix

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

A cached FashionMNIST subset contains real 28×28 grayscale clothing images and ten classes. The downloader validates the torchvision files; a generated-shape route runs only when the cache is absent.

**Small example:** A 3×3 edge filter slides across five image patches using the same nine weights. One learned edge detector can respond at the sleeve, collar, or shoe boundary.

In [ ]:
torch.manual_seed(SEED);cache=Path.cwd().parent/"_shared_data";transform=transforms.ToTensor()
try:
 full_train=datasets.FashionMNIST(cache,train=True,download=False,transform=transform);full_test=datasets.FashionMNIST(cache,train=False,download=False,transform=transform);train_ds=Subset(full_train,list(range(3000)));test_ds=Subset(full_test,list(range(1000)));class_names=full_train.classes;source="real FashionMNIST cache"
except Exception:
 print("OFFLINE BANNER: FashionMNIST cache absent; using generated stripe/shape images.");rng=np.random.default_rng(SEED)
 def synth(n):
  ims=np.zeros((n,1,28,28),np.float32);lab=rng.integers(0,10,n)
  for j,c in enumerate(lab): ims[j,0,2+c*2:4+c*2,:]=1;ims[j]+=rng.normal(0,.12,ims[j].shape)
  return TensorDataset(torch.tensor(ims),torch.tensor(lab))
 train_ds=synth(3000);test_ds=synth(1000);class_names=[f"shape {i}" for i in range(10)];source="synthetic fallback"
print("Source:",source,"train/test:",len(train_ds),len(test_ds),"sample:",train_ds[0][0].shape)

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
ims=torch.stack([train_ds[i][0] for i in range(20)]);labs=np.array([int(train_ds[i][1]) for i in range(len(train_ds))]);fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].imshow(np.hstack([ims[i,0] for i in range(8)]),cmap="gray");axes[0].axis("off");axes[0].set_title("Real examples" if "real" in source else "Fallback examples")
axes[1].bar(range(10),np.bincount(labs,minlength=10),color="#176b66");axes[1].set_title("Training class counts");axes[2].hist(ims.numpy().ravel(),bins=25,color="#d38b45");axes[2].set_title("Pixel intensities");plt.tight_layout();plt.show()

**Takeaway:** Local edges and textures recur at different positions.

**Takeaway:** The subset remains approximately class-balanced.

**Takeaway:** Most pixels are dark, so accuracy cannot come from treating every location equally informative.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
generator=torch.Generator().manual_seed(SEED);train_loader=DataLoader(train_ds,batch_size=64,shuffle=True,generator=generator,num_workers=0);test_loader=DataLoader(test_ds,batch_size=128,shuffle=False,num_workers=0);xb,yb=next(iter(train_loader));print("Batch tensors:",xb.shape,yb.shape)

# 6. Train

        Training turns the assumptions behind cnn image classification into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Convolve shared 3×3 filters over the image.
2. Apply ReLU and pooling to retain strong local responses.
3. Repeat to combine edges into motifs.
4. Flatten pooled maps and optimize class cross-entropy.

In [ ]:
net=nn.Sequential(nn.Conv2d(1,8,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(8,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Flatten(),nn.Linear(16*7*7,10));opt=torch.optim.Adam(net.parameters(),lr=.002);history=[]
for epoch in range(2):
 net.train();total=0
 for xb,yb in train_loader:
  opt.zero_grad();loss=nn.functional.cross_entropy(net(xb),yb);loss.backward();opt.step();total+=loss.item()*len(xb)
 history.append(total/len(train_ds));print(f"epoch {epoch+1}: loss={history[-1]:.3f}")

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
net.eval();truth=[];pred=[]
with torch.no_grad():
 for xb,yb in test_loader: truth.extend(yb.numpy());pred.extend(net(xb).argmax(1).numpy())
acc=accuracy_score(truth,pred);base=max(np.bincount(labs))/len(labs);cm=confusion_matrix(truth,pred);fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot([1,2],history,marker="o");axes[0].set(title="Training loss",xlabel="epoch")
sns.heatmap(cm,xticklabels=False,yticklabels=False,cmap="mako",cbar=False,ax=axes[1]);axes[1].set_title(f"Confusion · accuracy {acc:.3f}")
with torch.no_grad(): kernels=net[0].weight[:,0].numpy();axes[2].imshow(np.hstack(kernels),cmap="coolwarm");axes[2].set_title("Learned first-layer filters");axes[2].axis("off");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print(f"Accuracy={acc:.3f}; majority baseline={base:.3f}")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: A CNN assumes nearby pixels and repeated local patterns matter; a fixed pixel permutation destroys that locality. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
perm=torch.randperm(28*28,generator=torch.Generator().manual_seed(SEED));bad_truth=[];bad_pred=[]
with torch.no_grad():
 for xb,yb in test_loader:
  shuffled=xb.view(len(xb),-1)[:,perm].view_as(xb);bad_truth.extend(yb.numpy());bad_pred.extend(net(shuffled).argmax(1).numpy())
print(f"Normal accuracy={acc:.3f}; fixed pixel-shuffle accuracy={accuracy_score(bad_truth,bad_pred):.3f}")

**Use this when…** labels depend on local spatial motifs and enough images exist.

        **Don't use this when…** spatial layout is meaningless, data are tiny without transfer, or compute/latency cannot support convolution.

        ## Try this

        1. Add data augmentation with small translations.
2. Remove pooling and compare parameters/runtime.
3. Visualize which test images are confused most often.